In [ ]:
# ==========================================
# Import Necessary Libraries
# ==========================================
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)
import evaluate

# ==========================================
# Task 1: Dataset Selection
# ==========================================
print("--- Task 1: Dataset Selection ---")
print("Dataset: CoNLL-2003")
# We load the CoNLL-2003 dataset. It contains tokens, POS tags, chunk tags, and NER tags.
raw_datasets = load_dataset("conll2003")

# Extract the features for chunking
chunk_features = raw_datasets["train"].features["chunk_tags"]
label_names = chunk_features.feature.names

print(f"Label Categories (Chunk Tags): {label_names}")

# Create mappings for the model configuration
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

# ==========================================
# Task 2: Data Preprocessing & Label Alignment
# ==========================================
print("\n--- Task 2: Data Preprocessing ---")
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    """
    Tokenizes inputs and aligns the labels with the new subword tokens.
    BERT uses subword tokenization (e.g., 'playing' -> 'play', '##ing').
    We must label the first subword with the original tag, and assign -100
    to subsequent subwords so PyTorch ignores them during loss calculation.
    """
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []

    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens (like [CLS] and [SEP]) get mapped to None. Set them to -100.
            if word_idx is None:
                label_ids.append(-100)
            # First subword of a word gets the actual label
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Subsequent subwords get -100 so they are ignored in the loss function
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply the preprocessing to the dataset
# Using a subset to ensure it trains quickly in Colab without crashing
small_train_dataset = raw_datasets["train"].select(range(3000))
small_eval_dataset = raw_datasets["validation"].select(range(500))

tokenized_train = small_train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_eval = small_eval_dataset.map(tokenize_and_align_labels, batched=True)

# Use a data collator to automatically pad inputs and labels dynamically
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# ==========================================
# Task 3 & 5: Model Setup and Evaluation Metric
# ==========================================
print("\n--- Task 3 & 5: Model Setup & Evaluation Metric ---")
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
    num_labels=len(label_names)
)

metric = evaluate.load("seqeval")

def compute_metrics(p):
    """
    Computes Precision, Recall, and F1 Score using seqeval.
    Ignores the -100 padding tokens.
    """
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100) and convert to string labels
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# ==========================================
# Task 4: Training
# ==========================================
print("\n--- Task 4: Training ---")
training_args = TrainingArguments(
    output_dir="./chunking_model",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none" # Disables W&B logging prompt
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()

# ==========================================
# Task 6: Inference
# ==========================================
print("\n--- Task 6: Inference ---")
# Create a pipeline using our newly trained model
chunker = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

custom_sentence = "John works at Google in California."
print(f"Input: {custom_sentence}")

predictions = chunker(custom_sentence)
for pred in predictions:
    print(f"Word/Phrase: '{pred['word']}' | Chunk Tag: {pred['entity_group']} | Confidence: {pred['score']:.4f}")